# 06a · Separating a sentence into situation and phrase

The custom mechanism (situation classifier + phrase classifier + rule) needs the two parts of a sentence. In
dataset v2 they were given; in real tweets they are not, e.g.

> *Love waiting 2 hours for the bus* → phrase **Love**, situation **waiting 2 hours for the bus**.

Three segmenters (all may answer "not found" for a part):

| | segmenter | size | training |
|---|---|---|---|
| a | **rule**: sentiment cues (VADER lexicon + a short list of multi-word cues) and punctuation | 0 | none, fully transparent |
| b | **extractive QA** (DistilBERT-SQuAD): *What happened?* / *How does the writer feel about it?* | 66 M | none |
| c | **token classifier** (DistilBERT): every word → O / situation / phrase | 66 M | synthetic sentences |

**Synthetic sentences:** the 140 situations of v2 in ~30 templates (phrase before, after or on both sides of the
situation, plus sentences with only one part). The test part uses templates, situations and phrases that the
classifier never saw.

**Evaluation:** (1) synthetic test sentences; (2) 50 tweets of the iSarcasmEval test set annotated by hand
(`data/isarcasm_annotated_50.csv`: 25 sarcastic, 25 not). Finally all segmenters are applied to the whole
iSarcasmEval test set (1400 tweets) and to the 200 pairs of task C; the result is saved for 06b.

**Runtime:** L4 / T4 ≈ 10 min (training ≈ 2–3 min). CPU works but training takes ≈ 30 min.

## Setup

In [1]:
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q vaderSentiment
ISARC = "/content/iSarcasmEval"
if not os.path.exists(ISARC):
    !git clone -q https://github.com/iabufarha/iSarcasmEval.git {ISARC}

In [2]:
import os, json, time, numpy as np, pandas as pd, torch
from IPython.display import display
from src.prompts import load_dataset
from src.segment import (make_synthetic, evaluate, labels_from_spans, words, LexiconRule, QASegmenter,
                         TokenSegmenter, load_isarcasm, CATEGORIES)
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
N_PER_TEMPLATE = 60      # synthetic sentences per training template
EPOCHS = 3
SEED = 0
# ------------------------------------------
SUB = "06a_segmenter"
mount_drive()
print("results go to:", results_dir(SUB), "| GPU:", torch.cuda.is_available())
set_seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
results go to: /content/drive/MyDrive/sarcasm/06a_segmenter | GPU: True


## 1 · Synthetic sentences

In [3]:
v2 = load_dataset("v2")
SYN = make_synthetic(v2, v2, n_per_template=N_PER_TEMPLATE, seed=SEED)
print(SYN.groupby(["split", "family"]).size().unstack(), "\n")
print("templates:", SYN.groupby("split").template.nunique().to_dict(),
      "| situations:", SYN.groupby("split").situation.nunique().to_dict())
for t, lab in SYN.sample(10, random_state=1)[["text", "labels"]].values:
    ws = [w for w, _, _ in words(t)]
    print(" ".join(f"[{w}]" if l == "PHR" else (f"<{w}>" if l == "SIT" else w) for w, l in zip(ws, lab)))
print("\n[phrase]  <situation>")
SYN.drop(columns=["labels"]).to_csv(os.path.join(results_dir(SUB), "synthetic.csv"), index=False)
TR, TE = SYN[SYN.split == "train"], SYN[SYN.split == "test"]

family  both  only_phr  only_sit  post  pre
split                                      
test      20        20        20    60   40
train    120        60       120   420  360 

templates: {'test': 8, 'train': 18} | situations: {'test': 42, 'train': 98}
[So] [glad] [about] <my> <package> <getting> <lost> lol
[Dreading] <a> <muddy> <walk> today
<A> <late> <train> this morning . [Oh] [,] [how] [wonderful] [.]
<Relaxing> <on> <the> <beach> . [Honestly] [,] [it] [was] [awful] [.]
[Hate] <a> <loud> <night> !
[Yay] [for] <a> <clean> <kitchen> 🙃
Just <a> <sick> <dog> . . .
<My> <train> <arriving> <early> . . . [just] [what] [I] [needed]
[Great] [,] [just] [what] [I] [needed] [.]
Spent the afternoon <a> <quiet> <evening> <at> <home> .

[phrase]  <situation>


## 2 · The three segmenters

In [4]:
SEG = {"a · rule": LexiconRule()}

t0 = time.time()
qa = QASegmenter()
fit = TR.sample(min(400, len(TR)), random_state=SEED)
acc = qa.fit_threshold(fit.text, fit.has_both)
SEG["b · QA"] = qa.batch
print(f"QA model loaded ({time.time() - t0:.0f} s); minimum answer score chosen on training sentences: "
      f"{qa.min_score} (accuracy of 'both found': " + ", ".join(f"{k}: {v:.2f}" for k, v in acc.items()) + ")")

path = os.path.join(results_dir(SUB), "token_classifier")
if os.path.exists(os.path.join(path, "config.json")):
    tc = TokenSegmenter(path)
    print("token classifier loaded from", path)
else:
    tc = TokenSegmenter()
    t0 = time.time()
    tc.train(TR.text.tolist(), TR.labels.tolist(), epochs=EPOCHS, seed=SEED)
    tc.save(path)
    print(f"token classifier trained in {(time.time() - t0) / 60:.1f} min, saved to", path)
SEG["c · token classifier"] = tc.batch

def run(name, texts):
    f = SEG[name]
    return f(list(texts)) if name != "a · rule" else [f(t) for t in texts]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

... (8 outputs left out)

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/content/sarcasm-proposal/src/segment.py:431: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  tot 

  epoch 1: loss 0.3129
  epoch 2: loss 0.0039
  epoch 3: loss 0.0018


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

token classifier trained in 0.1 min, saved to /content/drive/MyDrive/sarcasm/06a_segmenter/token_classifier


## 3 · Evaluation on synthetic sentences (unseen templates, situations and phrases)

Word-level precision / recall / F1 for the situation and the phrase; "both found" compares whether a segmenter
finds both parts with whether the sentence has both.

In [5]:
rows, rows_f = {}, []
for name in SEG:
    pred = run(name, TE.text)
    rows[name] = evaluate(TE.text.tolist(), TE.labels.tolist(), pred)
    for fam in TE.family.unique():
        m = (TE.family == fam).values
        e = evaluate(TE.text[m].tolist(), TE.labels[m].tolist(), [p for p, k in zip(pred, m) if k])
        rows_f.append({"segmenter": name, "family": fam, "SIT F1": e["SIT F1"], "PHR F1": e["PHR F1"],
                       "both found: accuracy": e["both found: accuracy"]})
SYN_EVAL = pd.DataFrame(rows).T
display(SYN_EVAL.round(3))
display(pd.DataFrame(rows_f).pivot(index="family", columns="segmenter").round(2))

,SIT precision,SIT recall,SIT F1,PHR precision,PHR recall,PHR F1,both found: accuracy,both found: recall,both found: false alarms
a · rule,0.749,0.690,0.718,0.785,0.646,0.709,0.869,0.933,0.325
b · QA,0.810,0.223,0.349,0.371,0.554,0.444,0.431,0.317,0.225
c · token classifier,1.000,0.985,0.992,0.844,0.997,0.914,0.994,1.000,0.025


SIT F1                               PHR F1         \
segmenter a · rule b · QA c · token classifier a · rule b · QA   
family                                                           
both          0.78   0.72                 1.00     0.61   0.45   
only_phr      0.00   0.00                 0.00     0.68   0.90   
only_sit      0.21   0.06                 0.99     0.00   0.00   
post          0.76   0.37                 0.99     0.71   0.39   
pre           0.82   0.17                 0.98     0.80   0.48   

                               both found: accuracy         \
segmenter c · token classifier             a · rule b · QA   
family                                                       
both                      1.00                 1.00   0.65   
only_phr                  0.95                 0.60   0.75   
only_sit                  0.00                 0.75   0.80   
post                      0.92                 0.95   0.32   
pre                       0.83                 0.88   0.15   

                                
segmenter c · token classifier  
family                          
both                      1.00  
only_phr                  1.00  
only_sit                  0.95  
post                      1.00  
pre                       1.00

## 4 · Evaluation on 50 hand-annotated tweets

Annotation (`data/isarcasm_annotated_50.csv`): the phrase and the situation as exact substrings, empty when the
tweet has no such structure (e.g. satire, rhetorical questions, irony between two claims). Only 13 of the 25
sarcastic tweets have the structure — a first sign of how much of real sarcasm is "positive words about a
negative situation".

In [6]:
ANN = pd.read_csv("data/isarcasm_annotated_50.csv").fillna("")
def gold(t, s, p):
    return labels_from_spans(t, {"SIT": [(t.index(s), t.index(s) + len(s))] if s else [],
                                 "PHR": [(t.index(p), t.index(p) + len(p))] if p else []})
G = [gold(t, s, p) for t, s, p in zip(ANN.tweet, ANN.situation, ANN.phrase)]
print("tweets with both parts:", int(((ANN.situation != "") & (ANN.phrase != "")).sum()), "of", len(ANN),
      "| among the sarcastic:", int(((ANN.situation != "") & (ANN.phrase != "") & (ANN.sarcastic == 1)).sum()), "of 25")
ANN_PRED, rows = {}, {}
for name in SEG:
    ANN_PRED[name] = run(name, ANN.tweet)
    rows[name] = evaluate(ANN.tweet.tolist(), G, ANN_PRED[name])
ANN_EVAL = pd.DataFrame(rows).T
display(ANN_EVAL.round(3))

pd.set_option("display.max_colwidth", 70)
show = ANN[["tweet", "sarcastic", "phrase", "situation"]].copy()
for name, pr in ANN_PRED.items():
    k = name.split(" · ")[0]
    show[f"{k}: phrase"] = [p["phrase"] for p in pr]
    show[f"{k}: situation"] = [p["situation"] for p in pr]
display(show.head(25))

tweets with both parts: 17 of 50 | among the sarcastic: 13 of 25


,SIT precision,SIT recall,SIT F1,PHR precision,PHR recall,PHR F1,both found: accuracy,both found: recall,both found: false alarms
a · rule,0.289,0.598,0.389,0.507,0.459,0.482,0.60,0.824,0.515
b · QA,0.142,0.123,0.132,0.063,0.284,0.104,0.58,0.529,0.394
c · token classifier,0.207,0.492,0.291,0.107,0.608,0.182,0.38,0.882,0.879


,tweet,sarcastic,phrase,situation,a: phrase,a: situation,b: phrase,b: situation,c: phrase,c: situation
0,Breaking news: Pubs and bars the latest cure for COVID.,1,,,None,None,Breaking news: Pubs and bars the latest cure for COVID.,None,None,Pubs and bars the latest cure for COVID
1,I love it when my neighbours children block the drive because they...,1,I love it when,my neighbours children block the drive because they can’t cook the...,I love it when,my neighbours children block the drive because they can’t cook the...,love it when my neighbours children block the drive,None,I love it when … can’t,cook their own meals every single day
2,Isn't it just amazing how competent the government are. You know e...,1,Isn't it just amazing,how competent the government are,is so great,right now,everything is fine when we have a trustworthy PM who doesn't blata...,None,Isn't it just amazing how competent the … are. You know everything...,have a trustworthy PM
3,The Spanish National Ballet company rehearsing wearing masks: gasp...,1,,,protect,Makes sense that you'd refuse to wear a mask in a bus for five min...,Carry on,"gasping, falling about clutching their chests",Makes sense that you'd refuse to … Totally. Carry on.,The Spanish National Ballet company rehearsing wearing masks: gasping
4,I see ITV have new drama starting next week think it's about a pol...,1,,,problems that,i'll make change then,it's about a police detective with personal problems,None,see … that i'll make change then,ITV have new drama starting next week
5,"Max Verstappen is such a clean driver, he never makes dirty moves ...",1,is such a clean driver,Max Verstappen,dirty,moves when racing,he never makes dirty moves when racing,None,is such … he never makes dirty moves when,a clean driver
6,I love how Instagram can motivate people. I saw a post by Kim Kar...,1,I love how Instagram can motivate people,I was motivated to delete my account,I love how,Instagram can motivate people,motivated to delete my account,a post by Kim Kardashian and I was,can motivate people. I … and I was motivated to delete,saw a post by Kim Kardashian
7,It’s actually really easy to get into the mindset of a Tory. All I...,1,,,really easy to,get into the mindset of a Tory,slur incoherently,smash my head into a wall until I bleed,It’s actually really easy to get into the mindset of … All I do is...,social mobility and England before stuffing handfuls of loose chan...
8,I went to the best restaurant today for dinner. It was absolutely ...,1,It was absolutely amazing,My steak was cold and the fries were rock solid,the best,restaurant today for dinner,It was absolutely amazing,My steak was cold,were rock solid. I expected … and it sure was when I had to wash each,My steak was cold and the fries
9,Tips in hospitality during the festive season make it all worth it!,1,make it all worth it,Tips in hospitality during the festive season,the festive,season make it all worth it,Tips in hospitality,None,make it all worth it,Tips in hospitality during the festive season


## 5 · Segment the whole iSarcasmEval test set (for 06b)

1400 tweets (200 sarcastic) and the 200 pairs of task C (sarcastic tweet / the author's non-sarcastic rephrase).

In [7]:
TEST, PAIRS, TRAIN = load_isarcasm(ISARC)
print("test:", TEST.shape, "| sarcastic:", int(TEST.sarcastic.sum()), "| pairs:", PAIRS.shape, "| train:", TRAIN.shape)

def segment_all(texts):
    out = pd.DataFrame({"text": list(texts)})
    for name in SEG:
        k = name.split(" · ")[0]
        pr = run(name, out.text)
        out[f"{k}_phrase"] = [p["phrase"] for p in pr]
        out[f"{k}_situation"] = [p["situation"] for p in pr]
    return out

t0 = time.time()
SEG_TEST = cached("segments_test", lambda: segment_all(TEST.text), sub=SUB)
SEG_PAIRS0 = cached("segments_pairs_text0", lambda: segment_all(PAIRS.text_0), sub=SUB)
SEG_PAIRS1 = cached("segments_pairs_text1", lambda: segment_all(PAIRS.text_1), sub=SUB)
print(f"{time.time() - t0:.0f} s")

# coverage: share of tweets where both parts are found, by label and by category
rows = []
for k in ["a", "b", "c"]:
    both = SEG_TEST[f"{k}_phrase"].notna() & SEG_TEST[f"{k}_situation"].notna()
    r = {"segmenter": k, "all": both.mean(), "sarcastic": both[TEST.sarcastic == 1].mean(),
         "not sarcastic": both[TEST.sarcastic == 0].mean()}
    for c in CATEGORIES:
        r[c] = both[TEST[c] == 1].mean()
    rows.append(r)
COV = pd.DataFrame(rows).set_index("segmenter")
print("share of tweets with both parts found:")
display(COV.round(3))
print("category counts among the sarcastic tweets:", {c: int(TEST[c].sum()) for c in CATEGORIES})

test: (1400, 8) | sarcastic: 200 | pairs: (200, 3) | train: (3468, 9)
[saved]  /content/drive/MyDrive/sarcasm/06a_segmenter/segments_test.pt
[saved]  /content/drive/MyDrive/sarcasm/06a_segmenter/segments_pairs_text0.pt
[saved]  /content/drive/MyDrive/sarcasm/06a_segmenter/segments_pairs_text1.pt
6 s
share of tweets with both parts found:


,all,sarcastic,not sarcastic,sarcasm,irony,satire,understatement,overstatement,rhetorical_question
segmenter,,,,,,,,,
a,0.616,0.855,0.577,0.867,0.75,0.857,1.0,0.9,0.727
b,0.461,0.570,0.442,0.561,0.65,0.531,1.0,0.4,0.455
c,0.870,0.955,0.856,0.961,0.90,0.959,1.0,0.9,0.909


category counts among the sarcastic tweets: {'sarcasm': 180, 'irony': 20, 'satire': 49, 'understatement': 1, 'overstatement': 10, 'rhetorical_question': 11}


## 6 · Your own sentences

In [8]:
OWN = ["Love waiting 2 hours for the bus",
       "Thanks for the flat tire this morning, really made my day",
       "Monday morning meetings, just what I needed",
       "Oh great, another Monday"]
for t in OWN:
    print(t)
    for name in SEG:
        p = run(name, [t])[0]
        print(f"   {name:22s} phrase: {p['phrase']!s:35s} situation: {p['situation']}")

Love waiting 2 hours for the bus
   a · rule               phrase: Love                                situation: waiting 2 hours for the bus
   b · QA                 phrase: Love waiting 2 hours for the bus    situation: None
   c · token classifier   phrase: Love                                situation: waiting 2 hours for the bus
Thanks for the flat tire this morning, really made my day
   a · rule               phrase: really made my day                  situation: Thanks for the flat tire this morning
   b · QA                 phrase: Thanks for the flat tire            situation: None
   c · token classifier   phrase: Thanks for … really made my day     situation: the flat tire
Monday morning meetings, just what I needed
   a · rule               phrase: just what I needed                  situation: Monday morning meetings
   b · QA                 phrase: Monday morning meetings             situation: None
   c · token classifier   phrase: just what I needed                  

## Save

In [9]:
d = results_dir(SUB)
SYN_EVAL.to_csv(os.path.join(d, "eval_synthetic.csv"))
ANN_EVAL.to_csv(os.path.join(d, "eval_annotated.csv"))
show.to_csv(os.path.join(d, "annotated_predictions.csv"), index=False)
COV.to_csv(os.path.join(d, "coverage.csv"))
SEG_TEST.to_csv(os.path.join(d, "segments_test.csv"), index=False)
print("saved to", d)

saved to /content/drive/MyDrive/sarcasm/06a_segmenter
